<a href="https://colab.research.google.com/github/gowripreetham/SJSU_Deep_Learning_Advanced-customizations-in-deep-learning-and-neural-networks/blob/main/08_keras_custom_layers_and_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook 08: Keras Custom Layers and Custom Model

**Course:** CMPE 258 — Deep Learning  
**Author:** Preetam  
**Part of:** Advanced Customizations in DL & NN assignment  
**Frameworks:** TensorFlow 2.16 / Keras 3.3  
**Companion video:** TBD

## What this notebook covers
- Lambda/custom layers (`MyDense`, `AddGaussianNoise`, `MyLayerNormalization`)
- Subclassed `ResidualBlock` and `ResidualRegressor`
- Save/reload check for custom object serialization

## Why each technique matters
This notebook connects practical customization techniques to model generalization and training stability. Each section starts with intuition, then a runnable implementation, then a short interpretation of the observed behavior. Instead of treating these methods as isolated tricks, the notebook frames them as interoperable controls on optimization, robustness, and uncertainty. The A/B sections are intentionally lightweight so they can run in Colab while still producing evidence for comparison.


In [ ]:
!pip -q install tensorflow==2.16.1 keras==3.3.3 scikit-learn
import sys, platform
print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())



[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


Python: 3.11.9
Platform: macOS-26.0.1-arm64-arm-64bit


In [ ]:
# Set deterministic seeds for reproducibility.
import os
import random
import numpy as np

SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)

import tensorflow as tf
import keras

tf.random.set_seed(SEED)
keras.utils.set_random_seed(SEED)


California Housing keeps focus on architecture customization and serialization behavior in subclassed models.


## Simple custom layers


In [ ]:
import keras
import tensorflow as tf
from keras import layers
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
import numpy as np

X, y = fetch_california_housing(return_X_y=True)
X = X.astype("float32")
y = y.astype("float32")
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=42)

exponential_layer = layers.Lambda(lambda x: tf.exp(x))

class MyDense(layers.Layer):
    def __init__(self, units, **kwargs):
        super().__init__(**kwargs)
        self.units = units
    def build(self, input_shape):
        self.w = self.add_weight(shape=(input_shape[-1], self.units), initializer="glorot_uniform", trainable=True)
        self.b = self.add_weight(shape=(self.units,), initializer="zeros", trainable=True)
    def call(self, inputs):
        return tf.matmul(inputs, self.w) + self.b
    def get_config(self):
        cfg = super().get_config()
        cfg.update({"units": self.units})
        return cfg

class AddGaussianNoise(layers.Layer):
    def __init__(self, stddev=0.05, **kwargs):
        super().__init__(**kwargs)
        self.stddev = stddev
    def call(self, inputs, training=False):
        if training:
            return inputs + tf.random.normal(tf.shape(inputs), stddev=self.stddev)
        return inputs
    def get_config(self):
        cfg = super().get_config()
        cfg.update({"stddev": self.stddev})
        return cfg

class MyLayerNormalization(layers.Layer):
    def __init__(self, eps=1e-5, **kwargs):
        super().__init__(**kwargs)
        self.eps = eps
    def build(self, input_shape):
        dim = input_shape[-1]
        self.alpha = self.add_weight(shape=(dim,), initializer="ones", trainable=True)
        self.beta = self.add_weight(shape=(dim,), initializer="zeros", trainable=True)
    def call(self, x):
        mean = tf.reduce_mean(x, axis=-1, keepdims=True)
        var = tf.reduce_mean(tf.square(x - mean), axis=-1, keepdims=True)
        norm = (x - mean) / tf.sqrt(var + self.eps)
        return self.alpha * norm + self.beta


## ResidualBlock and ResidualRegressor subclassed model


In [ ]:
class ResidualBlock(layers.Layer):
    def __init__(self, units, n_layers=2, **kwargs):
        super().__init__(**kwargs)
        self.units = units
        self.n_layers = n_layers
        self.hidden = [layers.Dense(units, activation="relu") for _ in range(n_layers)]
    def call(self, x):
        h = x
        for layer in self.hidden:
            h = layer(h)
        return x + h
    def get_config(self):
        cfg = super().get_config()
        cfg.update({"units": self.units, "n_layers": self.n_layers})
        return cfg

class ResidualRegressor(keras.Model):
    def __init__(self, units=32, n_blocks=2, **kwargs):
        super().__init__(**kwargs)
        self.input_dense = layers.Dense(units, activation="relu")
        self.blocks = [ResidualBlock(units, n_layers=2) for _ in range(n_blocks)]
        self.noise = AddGaussianNoise(0.02)
        self.norm = MyLayerNormalization()
        self.out = layers.Dense(1)
        self.units = units
        self.n_blocks = n_blocks
    def call(self, x, training=False):
        x = self.input_dense(x)
        x = self.noise(x, training=training)
        for block in self.blocks:
            x = block(x)
        x = self.norm(x)
        return self.out(x)
    def get_config(self):
        cfg = super().get_config()
        cfg.update({"units": self.units, "n_blocks": self.n_blocks})
        return cfg

model = ResidualRegressor(units=32, n_blocks=2)
model.compile(optimizer="adam", loss="mse", metrics=["mae"])
h = model.fit(Xtr, ytr, validation_data=(Xte, yte), epochs=5, batch_size=64, verbose=0)
print("Final val_mse:", h.history["val_loss"][-1])


Final val_mse: 0.6546951532363892


## Save and reload custom model


In [ ]:
path = "residual_regressor.keras"
model.save(path)
reloaded = keras.models.load_model(
    path,
    custom_objects={
        "ResidualRegressor": ResidualRegressor,
        "ResidualBlock": ResidualBlock,
        "AddGaussianNoise": AddGaussianNoise,
        "MyLayerNormalization": MyLayerNormalization,
    },
)
pred1 = model.predict(Xte[:8], verbose=0)
pred2 = reloaded.predict(Xte[:8], verbose=0)
print("Reload max abs diff:", float(np.max(np.abs(pred1 - pred2))))


Reload max abs diff: 0.0


## Final summary table

| Technique | Validation evidence | notes |
|---|---|---|
| Lambda custom layer | executable `exponential_layer` | Fastest path for simple transforms |
| `MyDense` subclass | full `build/call/get_config` | Explicit weight management |
| `AddGaussianNoise` | noise only during `training=True` | Useful regularization hook |
| `MyLayerNormalization` | numerically stable normalized output | Manual implementation clarity |
| ResidualRegressor | val MSE trend + save/load parity | Subclassed model portability |

Serialization checks are essential for custom classes; successful reload is the practical proof that `get_config` and custom object registration are done correctly.
